# Parte 2 — Clasificación de sentimiento multilingüe

El notebook se guarda en VS Code; el kernel de Colab ejecuta las celdas en un servidor remoto.

1. En **Select Kernel → Colab → New Colab Server**, elige una GPU (por ejemplo T4).
2. En el Explorador, haz clic derecho en la carpeta **data → Upload to Colab**. El notebook busca `/content/data/train.csv` en el servidor y `data/train.csv` cuando se ejecuta desde la raíz local.
3. Ejecuta las celdas en orden. El flujo incluye una prueba inicial, una época de entrenamiento y la generación del primer CSV. Reservamos `eval.csv` para generar las predicciones de Kaggle.

### Carga e inspección de los datos

Leemos `train.csv`, que contiene el identificador, el texto y el sentimiento esperado de cada reseña. Buscamos la carpeta `data` local o su copia en `/content/data` cuando ejecutamos en Colab.

La inspección encontró **20.466 reseñas**, con IDs únicos y sin textos faltantes. Hay 7.486 positivas, 7.381 negativas y 5.599 neutrales. Las neutrales son menos frecuentes, pero para esta referencia no cambiamos la distribución de clases.

In [2]:
from pathlib import Path

import pandas as pd

# Cambia esta ruta si subiste data a otro lugar del servidor.
rutas_data = [Path("data"), Path("/content/data")]
DATA_DIR = next(
    (ruta.resolve() for ruta in rutas_data if (ruta / "train.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No encontré train.csv. En el Explorador de VS Code, "
        "haz clic derecho en data → Upload to Colab. "
        "Comprueba que quedó en /content/data."
    )

print("Carpeta de datos:", DATA_DIR)
train = pd.read_csv(DATA_DIR / "train.csv")

print(f"Reseñas: {len(train):,}")
print("Columnas:", train.columns.tolist())
print("IDs duplicados:", train["id"].duplicated().sum())
print("Textos faltantes:", train["text"].isna().sum())

print("\nReseñas por clase:")
print(train["label"].value_counts(dropna=False))

print("\nTres ejemplos:")
for fila in train.sample(3, random_state=42).itertuples(index=False):
    print(f"\nEtiqueta: {fila.label}")
    print(fila.text)

Carpeta de datos: /content/data
Reseñas: 20,466
Columnas: ['id', 'text', 'label']
IDs duplicados: 0
Textos faltantes: 0

Reseñas por clase:
label
positivo    7486
negativo    7381
neutral     5599
Name: count, dtype: int64

Tres ejemplos:

Etiqueta: neutral
Lo pedí a mediados de año y llegó en tres días a mi casa. Lo uso casi a diario en la oficina para juntitas y llamadas. Ya han pasado un par de meses y lo tengo funcionando sin parar. El paquete trae el cable de carga incluido, aparte del manual y la caja normal. Pesa poco y lo guardo en el cajón del escritorio cuando no lo uso.

Etiqueta: positivo
Compré esto a inicios de mes y llegó en unos cuatro días a mi casa. La batería me resultó ser mal construida desde que la estrené. La uso en la oficina casi a diario, más que nada para trámites del trabajo. El manual de uso me pareció estupendo hasta el momento.

Etiqueta: negativo
Lo compre hace un par de meses en una tienda en linea y me llego en tres dias. La duracion de la carga es esp

### Partición de datos

Las 20.466 reseñas tienen IDs únicos y no presentan textos faltantes. Separamos el 80 % para entrenar y el 20 % para validar, con estratificación por sentimiento y semilla 42. Usaremos la misma validación para comparar con y sin aumentación.

La validación queda fuera de las actualizaciones del modelo. La semilla fija el reparto y la estratificación mantiene proporciones parecidas de sentimientos; no garantiza que ambos grupos tengan la misma mezcla de idiomas. El conjunto de evaluación de Kaggle se reserva para generar el envío.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42

train_fit, val = train_test_split(
    train,
    test_size=0.20,
    random_state=SEED,
    stratify=train["label"],
)

train_fit = train_fit.reset_index(drop=True)
val = val.reset_index(drop=True)

print(f"Entrenamiento: {len(train_fit):,} reseñas")
print(f"Validación: {len(val):,} reseñas")

proporciones = pd.DataFrame({
    "entrenamiento": train_fit["label"].value_counts(normalize=True),
    "validación": val["label"].value_counts(normalize=True),
})

display((proporciones * 100).round(2))

Entrenamiento: 16,372 reseñas
Validación: 4,094 reseñas


,entrenamiento,validación
label,,
positivo,36.57,36.59
negativo,36.07,36.05
neutral,27.36,27.36


### Verificar el entorno

La salida debe mostrar `cuda` y el nombre de la GPU para el primer ajuste. Cambiar de servidor puede requerir volver a subir los datos.

En esta ejecución se detectó una **Tesla T4** y PyTorch seleccionó `cuda`. PyTorch realiza los cálculos de la red; Transformers proporciona el modelo preentrenado y el tokenizador. El notebook está en el Mac, pero el código y los archivos bajo `/content` están en el servidor remoto.

In [4]:
import sys
from importlib.util import find_spec

print("Python del kernel:", sys.executable)
print("Transformers instalado:", find_spec("transformers") is not None)

if find_spec("torch") is None:
    raise RuntimeError("Selecciona el kernel remoto de Colab con GPU.")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Para entrenar, cambia a un servidor de Colab con GPU.")

Python del kernel: /usr/bin/python3
Transformers instalado: True
PyTorch: 2.11.0+cu130
Dispositivo: cuda
GPU: Tesla T4


### Tokenización

Usamos [DistilBERT multilingüe](https://huggingface.co/distilbert/distilbert-base-multilingual-cased) (Apache 2.0). Su tokenizador convierte las reseñas en IDs de palabras y fragmentos de palabras. Conservamos mayúsculas, tildes y puntuación.

La reseña de ejemplo produjo **88 tokens**, incluidos los especiales. Por ejemplo, `pe` y `##dí` representan fragmentos de «pedí». Un `input_id` identifica un token en el vocabulario: no es un peso como en TF-IDF. La secuencia de IDs conserva el orden del texto.

In [5]:
from transformers import AutoTokenizer

MODELO_BASE = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

texto = train_fit.iloc[0]["text"]
entrada = tokenizer(texto, return_token_type_ids=False)
ids = entrada["input_ids"]

print("Reseña:", texto)
print("Total de tokens:", len(ids))
display(pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(ids[:25]),
    "input_id": ids[:25],
    "attention_mask": entrada["attention_mask"][:25],
}))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Reseña: Lo pedí en oferta después de estar un buen rato comparando precios, llegó en un par de meses porque venía de lejos. Trae carga por un puerto USB tipo C, lo cual es cómodo porque ya tengo cables de ese en casa. Lo uso más que nada en las noches antes de dormir. Y bueno, el botón de encendido superó todas mis expectativas.
Total de tokens: 88


,token,input_id,attention_mask
0,[CLS],101,1
1,Lo,13069,1
2,pe,11161,1
3,##dí,26530,1
4,en,10110,1
5,oferta,56721,1
6,después,12884,1
7,de,10104,1
8,estar,16627,1
9,un,10119,1


### Preparar lotes

El límite de 160 tokens recortó solo 13 de 16.372 reseñas de entrenamiento (0,08 %) en el diagnóstico previo. Añadimos relleno solo hasta el texto más largo de cada lote. DistilBERT no necesita `token_type_ids`.

Las etiquetas se convierten en `negativo = 0`, `neutral = 1` y `positivo = 2`. Son identificadores, no una escala numérica de sentimiento. En el lote de prueba, `input_ids` y `attention_mask` tienen forma `(4, 115)`: cuatro reseñas y 115 posiciones por fila. La máscara marca con 0 el relleno. `labels` tiene forma `(4,)`: una etiqueta por reseña.

In [6]:
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

MAX_LENGTH = 160
label2id = {"negativo": 0, "neutral": 1, "positivo": 2}
id2label = {i: etiqueta for etiqueta, i in label2id.items()}


class ResenasDataset(Dataset):
    def __init__(self, tabla):
        self.entradas = tokenizer(
            tabla["text"].tolist(),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_token_type_ids=False,
        )
        self.etiquetas = tabla["label"].map(label2id).astype(int).tolist()

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, indice):
        ejemplo = {
            nombre: valores[indice]
            for nombre, valores in self.entradas.items()
        }
        ejemplo["labels"] = self.etiquetas[indice]
        return ejemplo


dataset_train = ResenasDataset(train_fit)
dataset_val = ResenasDataset(val)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
lote_prueba = collator([dataset_train[i] for i in range(4)])

for nombre, tensor in lote_prueba.items():
    print(nombre, tuple(tensor.shape))

input_ids (4, 115)
attention_mask (4, 115)
labels (4,)


### Probar el modelo antes de entrenar

Cargamos los pesos preentrenados y una salida nueva para nuestras tres clases. Las puntuaciones iniciales sirven para comprobar el formato, no para evaluar la calidad del clasificador.

La salida `(4, 3)` confirma que hay una puntuación por clase para cada reseña. Estas puntuaciones son *logits*; `softmax` las convierte en probabilidades. El aviso de pesos nuevos en el clasificador es esperado al adaptar un modelo de lenguaje a tres clases. La pérdida de **1,1002** corresponde únicamente al lote de prueba antes del ajuste.

In [7]:
from transformers import AutoModelForSequenceClassification, set_seed

if device.type != "cuda":
    raise RuntimeError("Conecta un kernel Colab con GPU y vuelve a ejecutar el diagnóstico.")

set_seed(SEED)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
).to(device)

lote_gpu = {nombre: tensor.to(device) for nombre, tensor in lote_prueba.items()}
modelo.eval()
with torch.no_grad():
    salida = modelo(**lote_gpu)

print("Forma de la salida:", tuple(salida.logits.shape))
print(f"Pérdida inicial: {salida.loss.item():.4f}")
display(pd.DataFrame(
    salida.logits.softmax(dim=1).cpu().numpy(),
    columns=[id2label[i] for i in range(3)],
).round(3))

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Forma de la salida: (4, 3)
Pérdida inicial: 1.1002


,negativo,neutral,positivo
0,0.343,0.322,0.335
1,0.348,0.319,0.332
2,0.340,0.329,0.330
3,0.339,0.327,0.334


### Primer ajuste: una época sin aumentación

Entrenamos DistilBERT con las **16.372 reseñas de entrenamiento**. Una época significa recorrerlas una vez: en este caso, 1.024 lotes de hasta 16 reseñas, en orden mezclado.

En cada lote, el modelo produce una puntuación por clase y calcula la **pérdida de entropía cruzada** usando las etiquetas correctas. La retropropagación calcula cómo influyen los parámetros en esa pérdida. AdamW usa esos gradientes para actualizar los pesos con una tasa de aprendizaje de `2e-5`. Ajustamos tanto el clasificador nuevo como el encoder preentrenado.

Usamos precisión mixta para reducir el uso de memoria en la GPU. La validación se ejecuta después del ajuste, sin gradientes ni actualizaciones: contamos los aciertos al elegir la clase con mayor puntuación.

In [8]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

BATCH_SIZE = 16

loader_train = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)
loader_val = DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

optimizador = AdamW(modelo.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")

modelo.train()
suma_perdida = 0.0
vistos = 0

for paso, lote in enumerate(loader_train, start=1):
    lote = {nombre: tensor.to(device) for nombre, tensor in lote.items()}
    optimizador.zero_grad(set_to_none=True)

    # Usamos precisión mixta para ahorrar memoria en la GPU.
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perdida = modelo(**lote).loss

    scaler.scale(perdida).backward()
    scaler.step(optimizador)
    scaler.update()

    n = lote["labels"].size(0)
    suma_perdida += perdida.item() * n
    vistos += n

    if paso % 100 == 0 or paso == len(loader_train):
        print(
            f"Lote {paso}/{len(loader_train)} | "
            f"Pérdida media: {suma_perdida / vistos:.4f}",
            flush=True,
        )

print("\nEvaluando en validación...", flush=True)
modelo.eval()
aciertos = 0
vistos_val = 0

with torch.inference_mode():
    for lote in loader_val:
        lote = {nombre: tensor.to(device) for nombre, tensor in lote.items()}
        etiquetas = lote.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo(**lote).logits

        aciertos += (logits.argmax(dim=1) == etiquetas).sum().item()
        vistos_val += etiquetas.size(0)

accuracy_val = aciertos / vistos_val
print(f"Accuracy de validación: {accuracy_val:.4f} ({aciertos}/{vistos_val})")

Lote 100/1024 | Pérdida media: 0.8407
Lote 200/1024 | Pérdida media: 0.7288
Lote 300/1024 | Pérdida media: 0.6777
Lote 400/1024 | Pérdida media: 0.6295
Lote 500/1024 | Pérdida media: 0.5899
Lote 600/1024 | Pérdida media: 0.5582
Lote 700/1024 | Pérdida media: 0.5309
Lote 800/1024 | Pérdida media: 0.5064
Lote 900/1024 | Pérdida media: 0.4865
Lote 1000/1024 | Pérdida media: 0.4708
Lote 1024/1024 | Pérdida media: 0.4686

Evaluando en validación...
Accuracy de validación: 0.8395 (3437/4094)


### Resultado de validación e historial de ejecuciones

La primera ejecución reportada obtuvo **3.433/4.094 = 0,83854** de accuracy y una pérdida media de entrenamiento de **0,4672**. Ese experimento produjo el primer envío, con **0,789 en Kaggle público**.

Las salidas actualmente guardadas en la celda anterior corresponden a una nueva ejecución: **3.437/4.094 = 0,83952**, con pérdida media de **0,4686**. No debemos confundir ambas ejecuciones ni atribuir la diferencia únicamente a la recarga. El notebook no permite aislar la causa de los cuatro aciertos de diferencia.

### Guardar el modelo y generar el primer envío

Guardamos los pesos, la configuración y el tokenizador. Luego usamos `eval.csv` únicamente para inferencia, con el mismo tokenizador y límite de 160 tokens. Asociamos las predicciones por `id` y respetamos el orden de `sample_submission.csv`, con 6.781 filas.

**Precaución al reproducir:** volver a ejecutar estas celdas puede sobrescribir los archivos del experimento 1. Las salidas históricas de Kaggle corresponden al CSV que se envió en ese momento, no necesariamente a un archivo regenerado posteriormente.

In [9]:
from pathlib import Path

# Guardar los pesos, la configuración de clases y el tokenizador.
ruta_modelo = Path("/content/modelos/distilbert_multilingue_epoca1")
modelo.save_pretrained(ruta_modelo)
tokenizer.save_pretrained(ruta_modelo)
print("Modelo guardado en:", ruta_modelo, flush=True)

evaluacion = pd.read_csv(DATA_DIR / "eval.csv")
plantilla = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(plantilla.columns) == ["id", "answer"]
assert evaluacion["id"].is_unique and plantilla["id"].is_unique
assert set(evaluacion["id"]) == set(plantilla["id"])
assert evaluacion["text"].notna().all()

modelo.eval()
predicciones = []

with torch.inference_mode():
    for inicio in range(0, len(evaluacion), BATCH_SIZE):
        textos = evaluacion["text"].iloc[inicio:inicio + BATCH_SIZE].tolist()
        lote = tokenizer(
            textos,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_token_type_ids=False,
            return_tensors="pt",
        ).to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo(**lote).logits

        predicciones.extend(logits.argmax(dim=1).cpu().tolist())

# Asociar cada predicción a su ID y respetar el orden de la plantilla.
pred_por_id = pd.Series(
    [id2label[i] for i in predicciones],
    index=evaluacion["id"],
)
submission = plantilla[["id"]].copy()
submission["answer"] = submission["id"].map(pred_por_id)

assert len(submission) == 6781
assert submission["answer"].isin(label2id).all()

ruta_csv = Path("/content/submissions/submission_distilbert_epoca1.csv")
ruta_csv.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(ruta_csv, index=False)

display(submission.head())
print("\nPredicciones por clase:")
print(submission["answer"].value_counts())
print(f"\nCSV listo: {ruta_csv} ({len(submission):,} filas)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en: /content/modelos/distilbert_multilingue_epoca1


,id,answer
0,0,negativo
1,1,neutral
2,2,negativo
3,3,neutral
4,4,negativo



Predicciones por clase:
answer
negativo    3365
positivo    2120
neutral     1296
Name: count, dtype: int64

CSV listo: /content/submissions/submission_distilbert_epoca1.csv (6,781 filas)


### Archivos del primer experimento

El primer CSV enviado contenía 3.416 predicciones negativas, 2.068 positivas y 1.297 neutrales. La salida actualmente guardada, después de repetir el entrenamiento y generar el CSV, muestra 3.365 negativas, 2.120 positivas y 1.296 neutrales. Los conteos describen predicciones, no su exactitud.

El modelo se guarda en `/content/modelos/distilbert_multilingue_epoca1` y el envío en `/content/submissions/submission_distilbert_epoca1.csv`. Son rutas del servidor de Colab; para conservarlos fuera de la sesión es necesario descargarlos o copiarlos a almacenamiento persistente.

## Experimento 2 — Aumentación mediante cambios de mayúsculas

El primer experimento obtuvo **0,8385 de accuracy en validación** y **0,789 en Kaggle público**, con una época de DistilBERT multilingüe sin aumentación.

En este experimento presentamos en minúsculas una cuarta parte de las reseñas de entrenamiento que contienen mayúsculas, conservando su etiqueta. Esta transformación simula una variación de escritura sin cambiar las palabras ni añadir ejemplos al entrenamiento.

Partiremos nuevamente del modelo preentrenado. Mantendremos la misma partición de validación, el límite de 160 tokens y la configuración de entrenamiento del primer experimento. Así podremos medir

In [10]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("La sesión está usando CPU.")

PyTorch: 2.11.0+cu130
GPU disponible: True
GPU: Tesla T4


### Preparación de la aumentación

Copiamos el entrenamiento original y transformamos en minúsculas una cuarta parte de las reseñas que realmente cambian al hacerlo, seleccionadas con semilla 42. Se modificaron **4.093 textos**, manteniendo las mismas 16.372 filas, identificadores y etiquetas.

La validación no se transforma. La hipótesis es reducir la dependencia de las mayúsculas en un modelo `cased`; el efecto se determina experimentalmente y no se presupone positivo.

In [11]:
# Copia independiente: conservamos intacto el entrenamiento original.
train_fit_aug = train_fit.copy()

textos_minusculas = train_fit["text"].str.lower()
candidatas = train_fit.index[
    train_fit["text"].ne(textos_minusculas)
]

indices_aug = candidatas.to_series().sample(
    frac=0.25,
    random_state=SEED,
).index

train_fit_aug.loc[indices_aug, "text"] = (
    textos_minusculas.loc[indices_aug]
)

# Los IDs, las etiquetas y la cantidad de reseñas deben conservarse.
assert train_fit_aug[["id", "label"]].equals(
    train_fit[["id", "label"]]
)
assert len(train_fit_aug) == len(train_fit)

dataset_train_aug = ResenasDataset(train_fit_aug)

print(f"Entrenamiento: {len(dataset_train_aug):,}")
print(f"Reseñas transformadas: {len(indices_aug):,}")
print(f"Validación sin modificar: {len(dataset_val):,}")

indice = indices_aug[0]
print("\nOriginal:")
print(train_fit.loc[indice, "text"])
print("\nVariante:")
print(train_fit_aug.loc[indice, "text"])

Entrenamiento: 16,372
Reseñas transformadas: 4,093
Validación sin modificar: 4,094

Original:
Hace un mes se lo compré a un familiar que cumplía años. Lo pedí durante una promoción y llegó a la casa en unos cuatro días, en caja sellada. Lo revisamos apenas llegó y todo estaba completo según la lista del paquete. Lo entregamos en la fiesta junto con otros regalos y hasta ahora sigue guardado donde lo pusieron, sin uso todavía.

Variante:
hace un mes se lo compré a un familiar que cumplía años. lo pedí durante una promoción y llegó a la casa en unos cuatro días, en caja sellada. lo revisamos apenas llegó y todo estaba completo según la lista del paquete. lo entregamos en la fiesta junto con otros regalos y hasta ahora sigue guardado donde lo pusieron, sin uso todavía.


### Entrenamiento del modelo aumentado

Cargamos un modelo nuevo desde los pesos preentrenados y entrenamos **una época** con los textos preparados. No continuamos desde el modelo sin aumentación. Usamos lotes de 16, AdamW con tasa `2e-5` y precisión mixta. Guardamos esta variante bajo un nombre distinto: `distilbert_minusculas_epoca1`.

In [12]:
from pathlib import Path

from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(SEED)

modelo_aug = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=3,
    label2id=label2id,
    id2label=id2label,
).to(device)

BATCH_SIZE = 16

loader_train_aug = DataLoader(
    dataset_train_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)

loader_val_aug = DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

optimizador_aug = AdamW(modelo_aug.parameters(), lr=2e-5)
scaler_aug = torch.amp.GradScaler("cuda")

modelo_aug.train()
suma_perdida_aug = 0.0
vistos_aug = 0

for paso, lote in enumerate(loader_train_aug, start=1):
    lote = {
        nombre: tensor.to(device)
        for nombre, tensor in lote.items()
    }
    optimizador_aug.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perdida_aug = modelo_aug(**lote).loss

    scaler_aug.scale(perdida_aug).backward()
    scaler_aug.step(optimizador_aug)
    scaler_aug.update()

    n = lote["labels"].size(0)
    suma_perdida_aug += perdida_aug.item() * n
    vistos_aug += n

    if paso % 100 == 0 or paso == len(loader_train_aug):
        print(
            f"Lote {paso}/{len(loader_train_aug)} | "
            f"Pérdida media: {suma_perdida_aug / vistos_aug:.4f}",
            flush=True,
        )

print("\nEvaluando sobre la validación original...", flush=True)

modelo_aug.eval()
aciertos_aug = 0
vistos_val_aug = 0

with torch.inference_mode():
    for lote in loader_val_aug:
        lote = {
            nombre: tensor.to(device)
            for nombre, tensor in lote.items()
        }
        etiquetas = lote.pop("labels")

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo_aug(**lote).logits

        aciertos_aug += (
            logits.argmax(dim=1) == etiquetas
        ).sum().item()
        vistos_val_aug += etiquetas.size(0)

accuracy_val_aug = aciertos_aug / vistos_val_aug
accuracy_val_base = 3433 / 4094

# Guardar el experimento 2 con un nombre distinto.
ruta_modelo_aug = Path(
    "/content/modelos/distilbert_minusculas_epoca1"
)
modelo_aug.save_pretrained(ruta_modelo_aug)
tokenizer.save_pretrained(ruta_modelo_aug)

display(pd.DataFrame([
    {
        "experimento": "1 — Sin aumentación",
        "accuracy_validacion": accuracy_val_base,
    },
    {
        "experimento": "2 — Minúsculas en 4.093 reseñas",
        "accuracy_validacion": accuracy_val_aug,
    },
]).set_index("experimento").round(5))

print(
    f"\nAciertos del experimento 2: "
    f"{aciertos_aug}/{vistos_val_aug}"
)
print(
    "Diferencia frente al primero:",
    f"{(accuracy_val_aug - accuracy_val_base) * 100:+.3f} puntos porcentuales",
)
print("Modelo guardado en el servidor:", ruta_modelo_aug)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Lote 100/1024 | Pérdida media: 0.8514
Lote 200/1024 | Pérdida media: 0.7390
Lote 300/1024 | Pérdida media: 0.6836
Lote 400/1024 | Pérdida media: 0.6380
Lote 500/1024 | Pérdida media: 0.5996
Lote 600/1024 | Pérdida media: 0.5664
Lote 700/1024 | Pérdida media: 0.5376
Lote 800/1024 | Pérdida media: 0.5124
Lote 900/1024 | Pérdida media: 0.4908
Lote 1000/1024 | Pérdida media: 0.4750
Lote 1024/1024 | Pérdida media: 0.4733

Evaluando sobre la validación original...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,accuracy_validacion
experimento,
1 — Sin aumentación,0.83854
2 — Minúsculas en 4.093 reseñas,0.81705



Aciertos del experimento 2: 3345/4094
Diferencia frente al primero: -2.149 puntos porcentuales
Modelo guardado en el servidor: /content/modelos/distilbert_minusculas_epoca1


### Resultado del experimento 2 — Minúsculas

La aumentación obtuvo **3.345/4.094 = 0,81705** de accuracy en la validación sin modificar. La pérdida media de entrenamiento fue **0,4733**.

La tabla del código compara contra la cifra histórica fija `3433/4094`: son **88 aciertos menos**, equivalentes a **−2,15 puntos porcentuales**. Frente a la ejecución sin aumentación actualmente guardada (`3437/4094`), son **92 aciertos menos**, aproximadamente **−2,25 puntos**. No modificamos los resultados del código: distinguimos ambas referencias.

Mantuvimos la partición, la arquitectura, el límite de tokens, el tamaño de lote, la tasa de aprendizaje y una época. El clasificador partió de nuevo del modelo preentrenado. Esta transformación fue una sustitución de escritura en parte del entrenamiento; no añadió reseñas ni pasos extra.

En esta prueba las minúsculas no ayudaron. Es una comparación con y sin aumentación, no una demostración de que toda aumentación sea perjudicial. La variación entre ejecuciones aconseja interpretar las diferencias con prudencia. Este modelo no fue el elegido para el segundo envío.

## Experimento 3 — Continuación sin aumentación

Como las minúsculas no mejoraron la validación, recuperamos un checkpoint del entrenamiento original para estudiar si una pasada adicional por los mismos datos aporta una mejora.

### Verificación del checkpoint

Cargamos `distilbert_multilingue_epoca1` y evaluamos sin actualizar sus parámetros. La recarga produjo **3.437/4.094 = 0,83952**, que coincide con la nueva ejecución del experimento 1 guardada en este notebook, aunque difiere de los 3.433 aciertos de la primera ejecución histórica. Para la continuación usamos **0,83952** como referencia inmediata.

In [13]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

candidatas_modelo = [
    Path("/content/modelos/distilbert_multilingue_epoca1"),
    Path("/content/distilbert_multilingue_epoca1"),
]

ruta_base = next(
    (
        ruta for ruta in candidatas_modelo
        if (ruta / "config.json").is_file()
        and (ruta / "model.safetensors").is_file()
    ),
    None,
)

if ruta_base is None:
    raise FileNotFoundError(
        "Sube la carpeta distilbert_multilingue_epoca1 a Colab. "
        "Si quedó en otra ubicación, añádela a candidatas_modelo."
    )

modelo_sin_aug = AutoModelForSequenceClassification.from_pretrained(
    ruta_base,
    local_files_only=True,
).to(device)

assert modelo_sin_aug.config.id2label == id2label

loader_val_base = DataLoader(
    dataset_val,
    batch_size=16,
    shuffle=False,
    collate_fn=collator,
)

modelo_sin_aug.eval()
aciertos_recarga = 0
total_recarga = 0

with torch.inference_mode():
    for lote in loader_val_base:
        lote = {
            nombre: tensor.to(device)
            for nombre, tensor in lote.items()
        }
        etiquetas = lote.pop("labels")

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo_sin_aug(**lote).logits

        aciertos_recarga += (
            logits.argmax(dim=1) == etiquetas
        ).sum().item()
        total_recarga += etiquetas.size(0)

print("Modelo cargado desde:", ruta_base)
print(
    f"Accuracy de recarga: "
    f"{aciertos_recarga / total_recarga:.5f} "
    f"({aciertos_recarga}/{total_recarga})"
)
print("Referencia original: 3433/4094 = 0.83854")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo cargado desde: /content/modelos/distilbert_multilingue_epoca1
Accuracy de recarga: 0.83952 (3437/4094)
Referencia original: 3433/4094 = 0.83854


### Una época adicional con un optimizador nuevo

Una época es una pasada por las 16.372 reseñas de entrenamiento. Conservamos los pesos del checkpoint y hacemos una segunda pasada **sin aumentación**; no añadimos las reseñas de validación al entrenamiento.

El checkpoint conserva los pesos, pero no se guardó el estado de AdamW. Por eso esta continuación reinicia el optimizador con tasa `2e-5` y no equivale exactamente a una ejecución continua de dos épocas con su estado original.

El orden de entrenamiento vuelve a mezclarse con semilla 42. Evaluamos al finalizar y guardamos la nueva versión como `distilbert_sin_aug_epoca2`, separada del checkpoint inicial.

In [14]:
from torch.optim import AdamW
from transformers import set_seed

set_seed(SEED)

accuracy_antes_continuacion = aciertos_recarga / total_recarga

loader_continuacion = DataLoader(
    dataset_train,  # Entrenamiento original, sin minúsculas.
    batch_size=16,
    shuffle=True,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(SEED),
)

optimizador_continuacion = AdamW(
    modelo_sin_aug.parameters(),
    lr=2e-5,
)
scaler_continuacion = torch.amp.GradScaler("cuda")

modelo_sin_aug.train()
perdida_acumulada = 0.0
ejemplos_vistos = 0

for paso, lote in enumerate(loader_continuacion, start=1):
    lote = {
        nombre: tensor.to(device)
        for nombre, tensor in lote.items()
    }
    optimizador_continuacion.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        perdida = modelo_sin_aug(**lote).loss

    scaler_continuacion.scale(perdida).backward()
    scaler_continuacion.step(optimizador_continuacion)
    scaler_continuacion.update()

    n = lote["labels"].size(0)
    perdida_acumulada += perdida.item() * n
    ejemplos_vistos += n

    if paso % 100 == 0 or paso == len(loader_continuacion):
        print(
            f"Lote {paso}/{len(loader_continuacion)} | "
            f"Pérdida media: {perdida_acumulada / ejemplos_vistos:.4f}",
            flush=True,
        )

print("\nEvaluando después de la época adicional...", flush=True)

modelo_sin_aug.eval()
aciertos_continuacion = 0
total_continuacion = 0

with torch.inference_mode():
    for lote in loader_val_base:
        lote = {
            nombre: tensor.to(device)
            for nombre, tensor in lote.items()
        }
        etiquetas = lote.pop("labels")

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo_sin_aug(**lote).logits

        aciertos_continuacion += (
            logits.argmax(dim=1) == etiquetas
        ).sum().item()
        total_continuacion += etiquetas.size(0)

accuracy_continuacion = aciertos_continuacion / total_continuacion

ruta_modelo_epoca2 = Path(
    "/content/modelos/distilbert_sin_aug_epoca2"
)
modelo_sin_aug.save_pretrained(ruta_modelo_epoca2)
tokenizer.save_pretrained(ruta_modelo_epoca2)

display(pd.DataFrame([
    {
        "etapa": "Antes de continuar — modelo recargado",
        "accuracy": accuracy_antes_continuacion,
    },
    {
        "etapa": "Después de una época adicional",
        "accuracy": accuracy_continuacion,
    },
]).set_index("etapa").round(5))

print(
    f"\nAciertos: {aciertos_continuacion}/{total_continuacion}"
)
print(
    "Cambio:",
    f"{(accuracy_continuacion - accuracy_antes_continuacion) * 100:+.3f}",
    "puntos porcentuales",
)
print("Modelo guardado en:", ruta_modelo_epoca2)

Lote 100/1024 | Pérdida media: 0.3151
Lote 200/1024 | Pérdida media: 0.3073
Lote 300/1024 | Pérdida media: 0.2973
Lote 400/1024 | Pérdida media: 0.2855
Lote 500/1024 | Pérdida media: 0.2755
Lote 600/1024 | Pérdida media: 0.2673
Lote 700/1024 | Pérdida media: 0.2605
Lote 800/1024 | Pérdida media: 0.2550
Lote 900/1024 | Pérdida media: 0.2489
Lote 1000/1024 | Pérdida media: 0.2468
Lote 1024/1024 | Pérdida media: 0.2475

Evaluando después de la época adicional...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,accuracy
etapa,
Antes de continuar — modelo recargado,0.83952
Después de una época adicional,0.86004



Aciertos: 3521/4094
Cambio: +2.052 puntos porcentuales
Modelo guardado en: /content/modelos/distilbert_sin_aug_epoca2


### Resultado del experimento 3 y segundo envío

La época adicional alcanzó **3.521/4.094 = 0,86004** de accuracy en validación, frente a **0,83952** antes de continuar: **84 aciertos adicionales y +2,05 puntos porcentuales**. La pérdida media de entrenamiento terminó en **0,2475**.

La mejora en validación respalda conservar esta versión entre las evaluadas. No garantiza ausencia de sobreajuste ni implica que una tercera época vaya a mejorar.

Para el segundo envío usamos este modelo sin actualizar sus pesos con `eval.csv`. El CSV contiene **6.781 filas**, con 2.863 predicciones negativas, 2.601 positivas y 1.317 neutrales. El orden de los IDs sigue la plantilla de Kaggle.

**Kaggle público reportado:** el segundo envío obtuvo **0,82023**, frente al **0,789** reportado para el primero; una mejora de aproximadamente **3,12 puntos porcentuales**. El público evalúa otras reseñas y el resultado privado sigue pendiente.

In [15]:
from pathlib import Path

evaluacion_epoca2 = pd.read_csv(DATA_DIR / "eval.csv")
plantilla_epoca2 = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(plantilla_epoca2.columns) == ["id", "answer"]
assert evaluacion_epoca2["id"].is_unique
assert plantilla_epoca2["id"].is_unique
assert set(evaluacion_epoca2["id"]) == set(plantilla_epoca2["id"])
assert evaluacion_epoca2["text"].notna().all()

modelo_sin_aug.eval()
predicciones_epoca2 = []

with torch.inference_mode():
    for inicio in range(0, len(evaluacion_epoca2), 16):
        textos = evaluacion_epoca2["text"].iloc[inicio:inicio + 16].tolist()

        lote = tokenizer(
            textos,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_token_type_ids=False,
            return_tensors="pt",
        ).to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = modelo_sin_aug(**lote).logits

        predicciones_epoca2.extend(
            logits.argmax(dim=1).cpu().tolist()
        )

pred_por_id_epoca2 = pd.Series(
    [id2label[i] for i in predicciones_epoca2],
    index=evaluacion_epoca2["id"],
)

submission_epoca2 = plantilla_epoca2[["id"]].copy()
submission_epoca2["answer"] = (
    submission_epoca2["id"].map(pred_por_id_epoca2)
)

assert len(submission_epoca2) == 6781
assert submission_epoca2["answer"].isin(label2id).all()
assert submission_epoca2["id"].equals(plantilla_epoca2["id"])

ruta_csv_epoca2 = Path(
    "/content/submissions/submission_distilbert_sin_aug_epoca2.csv"
)
ruta_csv_epoca2.parent.mkdir(parents=True, exist_ok=True)
submission_epoca2.to_csv(ruta_csv_epoca2, index=False)

pd.testing.assert_frame_equal(
    pd.read_csv(ruta_csv_epoca2),
    submission_epoca2,
)

display(submission_epoca2.head())
print("\nPredicciones por clase:")
print(submission_epoca2["answer"].value_counts())
print(f"\nCSV listo: {ruta_csv_epoca2}")
print(f"Filas verificadas: {len(submission_epoca2):,}")

,id,answer
0,0,negativo
1,1,neutral
2,2,negativo
3,3,neutral
4,4,negativo



Predicciones por clase:
answer
negativo    2863
positivo    2601
neutral     1317
Name: count, dtype: int64

CSV listo: /content/submissions/submission_distilbert_sin_aug_epoca2.csv
Filas verificadas: 6,781


### Respaldo del modelo fuera de Colab

El archivo de pesos es demasiado grande para la descarga directa que intentamos con la extensión de VS Code. Empaquetamos la carpeta completa del modelo —pesos, configuración y tokenizador— en un ZIP y lo dividimos en **ocho partes de hasta 64 MiB**.

Las partes se descargan desde Colab y se unen en el Mac en su orden numérico. Verificar el ZIP con `unzip -t` permite comprobar la integridad de la transferencia. Esto no reentrena ni modifica los pesos: solo empaqueta y transporta archivos.

In [16]:
from pathlib import Path
from shutil import make_archive

carpeta_modelo = Path("/content/modelos/distilbert_sin_aug_epoca2")

assert (carpeta_modelo / "model.safetensors").is_file(), (
    "No encontré el modelo. Comprueba que sigues en el mismo servidor."
)

archivo_zip = Path(make_archive(
    "/content/distilbert_sin_aug_epoca2",
    "zip",
    root_dir=carpeta_modelo.parent,
    base_dir=carpeta_modelo.name,
))

carpeta_partes = Path("/content/partes_epoca2")
carpeta_partes.mkdir(exist_ok=True)

numero_partes = 0

with archivo_zip.open("rb") as archivo:
    while bloque := archivo.read(64 * 1024 * 1024):
        numero_partes += 1
        parte = carpeta_partes / f"modelo_epoca2.part{numero_partes:03d}"
        parte.write_bytes(bloque)
        print(f"Creada: {parte.name}")

print(f"\nDescarga las {numero_partes} partes desde: {carpeta_partes}")

Creada: modelo_epoca2.part001
Creada: modelo_epoca2.part002
Creada: modelo_epoca2.part003
Creada: modelo_epoca2.part004
Creada: modelo_epoca2.part005
Creada: modelo_epoca2.part006
Creada: modelo_epoca2.part007
Creada: modelo_epoca2.part008

Descarga las 8 partes desde: /content/partes_epoca2


## Resumen de los experimentos

| Experimento | Entrenamiento | Accuracy de validación | Kaggle público reportado |
|---|---|---:|---:|
| 1 — Referencia histórica | Una época, sin aumentación | 0,83854 | 0,789 |
| 1 — Nueva ejecución y checkpoint recargado | Una época, sin aumentación | 0,83952 | No reportado por separado |
| 2 — Aumentación | Una época, 4.093 reseñas en minúsculas | 0,81705 | No enviado |
| 3 — Continuación | Una época adicional sin aumentación, AdamW reiniciado | **0,86004** | **0,82023** |

La mejor versión medida hasta ahora es la continuación sin aumentación. La prueba de minúsculas se conserva como evidencia experimental negativa.

**Trabajo pendiente:** analizar errores por idioma sobre la validación, documentar las fuentes y licencias del modelo y el uso de IA generativa, y continuar las iteraciones necesarias para la entrega. Una sola partición y pocas ejecuciones no bastan para demostrar que una decisión generaliza a todos los datos.